In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import joblib

from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
from scipy.special import expit

import pymc as pm
import pymc_bart as pmb
import arviz as az
import arviz_plots as azp

import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from sklearn.calibration import calibration_curve

RANDOM_SEED = 694973
np.random.seed(RANDOM_SEED)

# for reproducibility
print("pandas: "+pd.__version__)
print("numpy: "+np.__version__)
print("pymc: "+pm.__version__)
print("pymc bart: "+pmb.__version__)
print("arviz: "+az.__version__)

In [ ]:
df = pd.read_csv('data/shot_probs_data.csv')
df.head()

# This model was trained on a subset of the data, 25/26 season to reduce training time.

In [ ]:
bart_features = ['distance', 'angle', 'is_behind_backboard']

def make_dataset(df:pd.DataFrame, set_name:str):
    dff = df.loc[df['set_name']==set_name].drop(['set_name','is_made','shot_type','shot_type_numeric'],axis=1).reset_index(drop=True)
    target = df.loc[df['set_name']==set_name]['is_made'].values
    shot_name_vals = df.loc[df['set_name']==set_name]['shot_type'].values
    return dff, target, shot_name_vals

X_train_mm, y_train, shot_train = make_dataset(df, 'train')
X_val_mm, y_val, shot_val = make_dataset(df, 'val')
X_query_mm, y_query, shot_query = make_dataset(df, 'query')
X_test_mm, y_test, shot_test = make_dataset(df, 'test')

print(X_train_mm.shape, X_val_mm.shape, X_query_mm.shape, X_test_mm.shape)

In [ ]:
X_train_mm.head()

In [ ]:
def sample(
    model: pm.Model, draws: int = 1000, tune: int = 1000, chains: int = 4, random_seed: int = RANDOM_SEED
):
    """
    Fit model using MCMC.

    Parameters
    ----------
    model: pm.Model
        PyMC model object.
    draws : int
        Number of draws to keep from the sampling process.
    tune : int
        Number of tuning steps to take before sampling.
    chains : int
        Number of chains to sample.
    target_accept : float
        Target acceptance probability for step size adaptation.
    random_seed : int
        Seed for randomness.
    """
    with model:
        trace = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            random_seed=random_seed,
            cores=4,
            idata_kwargs={"log_likelihood": False}
        )
    return trace

def compute_log_likelihood(model: pm.Model, trace: az.InferenceData) -> None:
    """Wrapper to compute elemwise log_likelihood of model given InferenceData with posterior group
    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling
    """
    with model:
        pm.compute_log_likelihood(trace)
    return None

def sample_posterior_pred(model: pm.Model, trace: az.InferenceData) -> az.InferenceData:
    """Generates samples from the posterior predictive distribution for model checks

    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling

    Returns:
        az.InferenceData: An ArviZ InferenceData object containing the posterior predictive samples.
    """
    with model:
        spp = pm.sample_posterior_predictive(
            trace,
            extend_inferencedata=True,
            random_seed=RANDOM_SEED,
        )
    return spp

In [ ]:
coords = {
        "bart_variables":np.array(bart_features),
        "obs_id": np.arange(len(y_train))
    }
with pm.Model(coords=coords) as model:
    # mutable data
    shot_made = pm.Data("shot_made", y_train, dims=("obs_id",))
    bart_data = pm.Data("bart_data", X_train_mm.values, dims=("obs_id", "bart_variables"))
    # Bart
    mu = pmb.BART("mu", X=bart_data, Y=shot_made, m=30, dims=("obs_id",))
    # likelihood
    shot_lkhood = pm.Bernoulli("shot_lkhood", logit_p=mu, observed=shot_made, dims=("obs_id",))

In [ ]:
trace_bart = sample(model)

In [ ]:
azp.plot_convergence_dist(trace_bart, var_names="mu")

In [ ]:
def define_behind_backboard(df, angle=150):
    theta_rad = np.radians(angle / 2)
    # directly behind
    behind = (df['yLegacy'] <= -10) & (df['yLegacy'] >= -40)
    depth = -10 - df['yLegacy']
    max_width = 30 + (depth * np.tan(theta_rad))
    in_width_range = df['xLegacy'].abs() <= max_width
    is_behind = behind & in_width_range
    return is_behind.astype(int)

yLegacy  = np.linspace(-40, 500, 200)
xLegacy = np.linspace(-250, 250, 200)
X, Y = np.meshgrid(xLegacy, yLegacy)

df = np.column_stack([X.ravel(), Y.ravel()])
df = pd.DataFrame(df, columns=['xLegacy','yLegacy'])
df['distance'] = np.sqrt((df["xLegacy"]/10)**2 + (df["yLegacy"]/10)**2)
df['angle'] = np.degrees(np.arctan2(df['xLegacy'].abs(), df['yLegacy']))
df['is_behind_backboard'] = define_behind_backboard(df, angle=150)

num_cols = ['distance', 'angle']
mm_scaler = joblib.load('data/mm_scaler.pkl')

grid_mm = mm_scaler.transform(df[num_cols])
grid_mm = pd.DataFrame(grid_mm, columns=num_cols, index=df.index)
grid_mm['is_behind_backboard'] = df['is_behind_backboard']
q_list = np.linspace(0, 1, 50).tolist()

In [ ]:
axes = pmb.plot_pdp(
    mu, 
    X=X_train_mm, 
    Y=y_train, 
    sharey=False, 
    xs_interval="quantiles", 
    xs_values=q_list
) 
ax_dist, ax_angle = axes[0], axes[1] 
fig = plt.gcf() 
fig.texts.clear() 
for ax in [ax_dist, ax_angle]:
    for line in ax.lines:
        if (line.get_linestyle() in ['--']):
            line.remove()

ax_dist.set_title("Distance (feet)") 
ax_dist.set_ylabel("Log-odds", fontsize=8)
ax_dist.set_ylim(-1.5, 1.5)
ax_dist.grid(True)
ax_dist.set_yticks(np.arange(-1.5, 2.0, 0.5)) 

ax_angle.set_title("Angle (degrees)") 
ax_angle.set_ylabel("Log-odds", fontsize=8)
ax_angle.set_ylim(-0.3, 0.1) 
ax_angle.grid(True)
ax_angle.set_yticks(np.arange(-0.3, 0.2, 0.1)) 

for ax in [ax_dist, ax_angle]: 
    ax.axhline(0, color='black') 

plt.suptitle("BART PDP", fontsize=14)
plt.tight_layout()

In [ ]:
def make_preds(bart_data: pd.DataFrame, model:pm.model, trace: az.InferenceData, batch_size: int = 5000) -> dict:
    n_obs = bart_data.shape[0]
    batches = np.arange(0, n_obs, batch_size)

    logits_mean_list = []
    probs_mean_list = []

    for start in tqdm(batches):
        end = min(start + batch_size, n_obs)
        batch_df = bart_data.iloc[start:end]

        with model:
            pm.set_data({
                "shot_made": np.zeros(batch_df.shape[0], dtype="int32"),
                "bart_data": batch_df.values
            }, coords={"obs_id": np.arange(batch_df.shape[0])}
            )
            
            preds = pm.sample_posterior_predictive(
                trace,
                var_names=['mu'],
                random_seed=RANDOM_SEED,
                progressbar=False
            )

        logits = preds.posterior_predictive['mu'].values
        logits_mean = logits.mean(axis=(0, 1))
        probs = expit(logits)
        logits_mean_list.append(logits_mean)
        probs_mean_list.append(probs.mean(axis=(0,1)))

        del preds, logits, logits_mean, probs

    return {
        "logits": np.concatenate(logits_mean_list),
        "probs": np.concatenate(probs_mean_list),
    }

In [ ]:
train_trace = make_preds(X_train_mm, model, trace_bart)

In [ ]:
def compute_scores(target, preds):
    loss = log_loss(target, preds)
    auc = roc_auc_score(target, preds)
    binary_preds = (preds >= 0.5).astype(int)
    accuracy = accuracy_score(target, binary_preds)
    print(f"Log Loss: {loss:.4f}, AUC: {auc:.4f}, Accuracy: {accuracy:.4f}")
    return None

In [ ]:
compute_scores(y_train, train_trace['probs'])

In [ ]:
val_trace = make_preds(X_val_mm, model, trace_bart)

In [ ]:
yLegacy  = np.linspace(-40, 500, 200)
xLegacy = np.linspace(-250, 250, 200)
X, Y = np.meshgrid(xLegacy, yLegacy)

# scale
df = np.column_stack([X.ravel(), Y.ravel()])
df = pd.DataFrame(df, columns=['xLegacy','yLegacy'])
df['distance'] = np.sqrt((df["xLegacy"]/10)**2 + (df["yLegacy"]/10)**2)
df['angle'] = np.degrees(np.arctan2(df['xLegacy'].abs(), df['yLegacy']))
df['is_behind_backboard'] = define_behind_backboard(df, angle=150)

num_cols = ['distance', 'angle']
mm_scaler = joblib.load('data/mm_scaler.pkl')
grid_mm = mm_scaler.transform(df[num_cols])
grid_mm = pd.DataFrame(grid_mm, columns=num_cols, index=df.index)
grid_mm['is_behind_backboard'] = df['is_behind_backboard']

# make probs
grid_trace = make_preds(grid_mm, model, trace_bart)
Z = grid_trace['probs'].reshape(X.shape)